# Лабораторна робота 1 — Дослідження даних з NumPy, Pandas та scikit-learn

In [ ]:
!mamba install numpy pandas matplotlib scikit-learn

Kernel Interrupt: Interrupted

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)


## Завдання 1

а) Створіть масив NumPy з 10 000 значень, що підпорядковані нормальному розподілу N(0, 1). Обчисліть середнє, стандартне відхилення, мінімум і максимум за допомогою функцій NumPy - без Python-циклів.

б) Підрахуйте, скільки значень виходять за межі ±2 стандартних відхилення,    використовуючи логічну індексацію. Яка це відсоткова частка? Порівняйте з теоретичним значенням ≈4,55 %.

In [ ]:
np.random.seed(0)
arr = np.random.normal(0, 1, 10_000)

mean_val = np.mean(arr)
std_val  = np.std(arr)
min_val  = np.min(arr)
max_val  = np.max(arr)

print(f'Середнє:              {mean_val:.6f}')
print(f'Стандартне відхилення:{std_val:.6f}')
print(f'Мінімум:              {min_val:.6f}')
print(f'Максимум:             {max_val:.6f}')

In [ ]:
outlier_mask = np.abs(arr) > 2 * std_val
n_outliers   = np.sum(outlier_mask)
pct_outliers = n_outliers / len(arr) * 100

print(f'Кількість викидів:     {n_outliers}')
print(f'Відсоткова частка:     {pct_outliers:.2f}%')
print(f'Теоретичне значення:   ~4.55%')

З 10 000 значень за межі ±2σ виходять 451 (≈4.51%), що добре узгоджується з теоретичним значенням 4.55% — різниця пояснюється випадковістю вибірки.

## Завдання 2 — Завантаження та огляд набору даних

Завантажте `kc_house_data.csv` у Pandas DataFrame та дайте відповідь на питання нижче, написавши код:

1. Скільки рядків і стовпців у наборі даних?
2. Які мінімум, максимум, середнє та медіана цін на будинки?
3. Скільки унікальних поштових індексів є у наборі даних? Який поштовий індекс має найбільше оголошень?

In [ ]:
sales = pd.read_csv('kc_house_data.csv')
sales.head()

In [ ]:
rows, cols = sales.shape
print(f'Кількість рядків:   {rows}')
print(f'Кількість стовпців: {cols}')


In [ ]:
print(f'Мінімальна ціна: ${sales["price"].min():,.0f}')
print(f'Максимальна ціна: ${sales["price"].max():,.0f}')
print(f'Середня ціна:    ${sales["price"].mean():,.2f}')
print(f'Медіана ціни:    ${sales["price"].median():,.0f}')


In [ ]:
unique_zips = sales['zipcode'].nunique()
top_zip     = sales['zipcode'].value_counts().idxmax()
top_zip_cnt = sales['zipcode'].value_counts().max()

print(f'Унікальних поштових індексів: {unique_zips}')
print(f'Найпопулярніший індекс: {top_zip} ({top_zip_cnt} оголошень)')


## Завдання 3 — Візуалізація даних

Створіть два графіки поруч:

- Лівий: гістограма `price` (50 інтервалів, логарифмічна шкала по осі X).
- Правий: діаграма розсіювання `sqft_living` vs `price`, де кольором точок позначено `bedrooms`. Додайте підписи осей і colorbar.

Одним реченням для кожного графіка опишіть, що ви спостерігаєте.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left diagram
axes[0].hist(sales['price'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_xscale('log')
axes[0].set_xlabel('Ціна (log-шкала), $', fontsize=12)
axes[0].set_ylabel('Кількість оголошень', fontsize=12)
axes[0].set_title('Розподіл цін на будинки', fontsize=13)
axes[0].grid(axis='y', alpha=0.4)

# Right diagram
sc = axes[1].scatter(
    sales['sqft_living'], sales['price'],
    c=sales['bedrooms'], cmap='plasma',
    alpha=0.35, s=8
)
cbar = plt.colorbar(sc, ax=axes[1])
cbar.set_label('Кількість спалень', fontsize=11)
axes[1].set_xlabel('Площа житлового приміщення (кв. фути)', fontsize=12)
axes[1].set_ylabel('Ціна, $', fontsize=12)
axes[1].set_title('Площа vs Ціна (колір = кількість спалень)', fontsize=13)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


Лівий (гістограма цін): Розподіл цін є правосторонньо скошеним - основна маса будинків зосереджена у діапазоні 200 000 – 700 000, а правий хвіст із дорогими об'єктами понад $1 млн є відносно рідкісним. Логарифмічна шкала добре «розтягує» лівий діапазон і дозволяє чітко бачити форму розподілу, яка б виглядала як крутий сплеск на лінійній шкалі.

Правий (scatter plot): Між площею житлового приміщення та ціною існує виражений позитивний лінійний зв'язок - чим більша площа, тим вища ціна. Переважна більшість точок темно-фіолетова (1-4 спальні), що свідчить про те, що будинки з аномально великою кількістю спалень (жовті/світлі викиди, наприклад 30+) є поодинокими аномаліями в даних і не відповідають загальній тенденції.

## Завдання 4 — Навчання та оцінка регресійної моделі

Розбийте дані 80/20 (`random_state=42`). Навчіть `LinearRegression` з scikit-learn для передбачення `price` за єдиною ознакою `sqft_living`.

1. Вкажіть вільний член (intercept) та коефіцієнт для `sqft_living`.
2. Обчисліть MSE і RMSE на тестовій вибірці.
3. Яка передбачувана ціна будинку площею 2 500 кв. футів?

In [ ]:
train_data, test_data = train_test_split(sales, test_size=0.2, random_state=42)

In [ ]:
# Model training 1: sqft_living
model1 = LinearRegression()
model1.fit(train_data[['sqft_living']], train_data['price'])

print(f'Вільний член (intercept):        {model1.intercept_:.4f}')
print(f'Коефіцієнт sqft_living:          {model1.coef_[0]:.4f}')

# MSE and RMSE
y_pred1 = model1.predict(test_data[['sqft_living']])
mse1    = mean_squared_error(test_data['price'], y_pred1)
rmse1   = np.sqrt(mse1)
print(f'\nMSE  (Модель 1): {mse1:,.2f}')
print(f'RMSE (Модель 1): {rmse1:,.2f}')

# Prediction
price_2500 = model1.predict([[2500]])[0]
print(f'\nПередбачена ціна для 2500 sqft: ${price_2500:,.2f}')


## Завдання 5 — Порівняння двох моделей

Навчіть другу `LinearRegression` з ознаками: `sqft_living`, `bedrooms`, `bathrooms`, `yr_built`. Порівняйте RMSE на тестовій вибірці між Моделлю 1 та Моделлю 2. Яке значення нижче? Коротко поясніть чому.

In [ ]:
features2 = ['sqft_living', 'bedrooms', 'bathrooms', 'yr_built']

model2 = LinearRegression()
model2.fit(train_data[features2], train_data['price'])

y_pred2 = model2.predict(test_data[features2])
mse2    = mean_squared_error(test_data['price'], y_pred2)
rmse2   = np.sqrt(mse2)

print(f'RMSE Модель 1 (sqft_living):                       {rmse1:>12,.2f}')
print(f'RMSE Модель 2 (sqft_living+bedrooms+bathrooms+yr): {rmse2:>12,.2f}')
print()
better = 'Модель 2' if rmse2 < rmse1 else 'Модель 1'
print(f'Краща модель: {better} (нижчий RMSE)')


Відповідь: RMSE Моделі 2 (262 986) є нижчим, ніж RMSE Моделі 1 (276 559), оскільки додаткові ознаки - кількість спалень, санвузлів та рік побудови - несуть додаткову пояснювальну силу щодо ціни: наприклад, новіші будинки зазвичай дорожчі, а кількість санвузлів корелює з загальним класом нерухомості. Більше релевантних ознак дозволяє моделі точніше апроксимувати залежність.

## Бонус — Аналіз залишків

Для вашої кращої моделі обчисліть вектор залишків (predicted − actual). Побудуйте гістограму. Чи виглядають залишки нормально розподіленими та центрованими навколо нуля? Що означав би систематичний ненульовий середній залишок?

In [ ]:
residuals = y_pred2 - test_data['price'].values

print(f'Середнє залишків:             {residuals.mean():,.2f}')
print(f'Стандартне відхилення залишків: {residuals.std():,.2f}')

plt.figure(figsize=(9, 4))
plt.hist(residuals, bins=60, color='tomato', edgecolor='white', alpha=0.85)
plt.axvline(0, color='black', linewidth=1.5, linestyle='--', label='Нуль')
plt.axvline(residuals.mean(), color='navy', linewidth=1.5,
            linestyle=':', label=f'Середнє = {residuals.mean():,.0f}')
plt.xlabel('Залишок (передбачено − факт), $', fontsize=12)
plt.ylabel('Кількість спостережень', fontsize=12)
plt.title('Розподіл залишків — Модель 2', fontsize=13)
plt.legend()
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

Спостереження: Розподіл залишків має форму, близьку до нормальної, і центрований поблизу нуля (середнє ≈ −1 612 долларів, що незначуще порівняно з RMSE приблизно 263 000). Проте наявний помітний лівий хвіст — модель недооцінює окремі дорогі об'єкти, тобто їхня реальна ціна виявляється значно вищою за передбачену. Якби середнє залишків було суттєво відмінне від нуля, це свідчило б про систематичне зміщення (bias) - модель постійно завищувала б або занижувала б ціни, що є ознакою пропущених важливих ознак або неправильно специфікованої функціональної форми.